In [1]:

# If "Suisse Int’l" is the exact internal name, this should work:
# (Optional) You can explicitly register each font file with Matplotlib:
import matplotlib.font_manager as fm
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-Light.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-LightItalic.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-Medium.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-MediumItalic.ttf")


import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

import numpy as np


colors = {
    'pink_dark' : '#f75785',
    'pink_light' : '#f8b0be',
    'aqua_dark' : '#009da5',
    'aqua_light' : '#3cc5be',
    'orange_dark' : '#ffa631',
    'orange_light' : '#ffd766',
    'red_dark' : '#e84743',
    'red_light' : '#ed8e83',
    'blue_dark': '#489fee',
    'blue_light': '#8fcfff',
    'dark_grey': '#413d3a',
    'light_grey': '#cac7c7',
}
def create_custom_cmap(colors):
    """
    Creates a custom colormap that transitions:
      #FB4C59 -> #f2a02a -> #009da5
    and returns it as a LinearSegmentedColormap.
    """
    # Define our three anchor colors in hex
    sel_colors = [colors['aqua_dark'], colors['orange_dark'], colors['pink_dark']]
    # Create a colormap with these three points
    cmap = mcolors.LinearSegmentedColormap.from_list("my_custom_cmap", sel_colors)
    return cmap

colors_list = list(colors.values())



color_map = create_custom_cmap(colors)
x = np.linspace(0, 1, 100)
X, Y = np.meshgrid(x, x)
Z = X + Y


## 1. Import Target and Anchor maps

Later: we will put two or three example maps in a folder, for now we do the import from the pc. For the moment we have two lists of maps and link to their elements inside the folder input.

In [ ]:
# create the list of anchor and target maps 
import os 
import pandas as pd
import json
from modules.data_preparation import extract_epsg

MAIN_FOLDER = '/Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/'
map_folders = os.listdir(MAIN_FOLDER) # READ THE FOLDER PATH AND GET THE LIST OF THE FOLDERS INSIDE
map_folders = [x for x in map_folders if x != '.DS_Store'] # REMOVE THE .DS_STORE FILE
anchor_list = []
target_list = []

# Collecting the map files
for folder in map_folders:
    if os.path.exists(MAIN_FOLDER + folder + '/'):
        map_files = [x for x in os.listdir(MAIN_FOLDER + folder + '/') if x != '.DS_Store']  # REMOVE THE .DS_STORE FILE
        objects_to_collect = {}  # Dictionary to store the map subfiles to collect
        year = folder.split('_')[0]
        auth = folder.split('_')[1]
        name_parts = folder.split('_')
        
        if len(name_parts) == 3:
            try:
                int(name_parts[2])
                number = name_parts[2]
                name = auth + '_' + year + '_' + number
            except:
                name = auth + '_' + name_parts[2] + '_' + year
        else:
            name = auth + '_' + year
  
        for file in map_files:
            if file.endswith(name + '.png') or file.endswith(name + '.jpg') or file.endswith(name + '.tif') or file.endswith(name + '.pdf') or file.endswith(name + '.jpeg') or file.endswith(name + '.tiff'):
                image_path = MAIN_FOLDER + folder + '/' + file
                objects_to_collect['image_path'] = image_path
                
            if file.endswith('mask.png'):
                mask_path = MAIN_FOLDER + folder + '/' + file
                if os.path.exists(mask_path):
                    objects_to_collect['mask_path'] = mask_path
                
            if file.endswith('.points'):
                file_path = MAIN_FOLDER + folder + '/' + file
                
                first_row = pd.read_csv(file_path, nrows=1, delimiter=',')
                points = pd.read_csv(file_path, skiprows=1, delimiter=',') 
                
                if points is not None:
                    objects_to_collect['points'] = points
            
                crs_info = str(first_row.columns)
                epsg_code = extract_epsg(crs_info)  # Extract EPSG code using the new function       
                objects_to_collect['epsg'] = epsg_code
            
            objects_to_collect['folder'] = folder
            objects_to_collect['folder_path'] = MAIN_FOLDER + folder + '/'
            
                
            if file.endswith('metadata.json'):
                with open(MAIN_FOLDER + folder + '/' + file) as json_file:
                    try:
                        metadata = json.load(json_file)
                        objects_to_collect['metadata'] = metadata
                    except:
                        continue
        
        if 'points' in objects_to_collect.keys() and 'mask_path' in objects_to_collect.keys():
            # Create a 'processed' folder inside the map's folder
            # if all the objects are not None
            if [x for x in objects_to_collect.values() if x is None]:
                continue
            processed_folder = os.path.join(MAIN_FOLDER, folder, 'processed')
            os.makedirs(processed_folder, exist_ok=True)  # Create the processed folder if it doesn't exist

            # Determine the output path for the processed tensor
            processed_image_name = os.path.basename(objects_to_collect['image_path']).split('.')[0] + '_processed.pt'
            processed_image_path = os.path.join(processed_folder, processed_image_name)

            anchor_list.append(objects_to_collect)
        elif 'points' not in objects_to_collect.keys() and 'image_path' in objects_to_collect.keys():
            target_list.append(objects_to_collect)

print(f'{len(anchor_list)} anchor maps out of {len(map_folders)}')
print(f'{len(target_list)} target maps out of {len(map_folders)}')


# save them with pickle
import pickle
with open('input/anchor_maps.pkl', 'wb') as f:
    pickle.dump(anchor_list, f)

with open('input/target_maps.pkl', 'wb') as f:
    pickle.dump(target_list, f)

anchor_maps = anchor_list
target_maps = target_list


NameError: name 'epsg_code' is not defined

In [20]:
# read the list of anchor and target maps

import pickle
with open('./input/anchor_maps.pkl', 'rb') as f:
    anchor_maps = pickle.load(f)
with open('./input/target_maps.pkl', 'rb') as f:
    target_maps = pickle.load(f)



In [21]:
# create from scratch the MapDataset object
from modules.MapDataset import MapDataset, create_map_dataset
from tqdm import tqdm

# Creating a list of MapDataset objects from the collected map data with tqdm
anchor_map_objects_list = [create_map_dataset(map_data) for map_data in tqdm(anchor_maps, desc="Creating MapDataset objects")]


Creating MapDataset objects:   0%|          | 0/86 [00:00<?, ?it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1898_schneller' with author 'schneller' and year '1898'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1898_schneller/schneller_1898.jpeg
libpng warning: iCCP: known incorrect sRGB profile
INFO:modules.MapDataset:Loaded mask from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1898_schneller/schneller_1898_mask.png
INFO:modules.MapDataset:MapDataset created successfully for folder: 1898_schneller
Creating MapDataset objects:   1%|          | 1/86 [00:00<00:22,  3.80it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1860_weller' with author 'weller' and year '1860'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/Li

In [22]:
#from modules.MapDataset import MapDataset, create_map_dataset

target_map_objects_list = [create_map_dataset(map_data) for map_data in tqdm(target_maps, desc="Creating MapDataset objects")]  

Creating MapDataset objects:   0%|          | 0/113 [00:00<?, ?it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1866_unknown_2' with author 'unknown' and year '1866'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1866_unknown_2/unknown_1866_2.jpg
libpng warning: iCCP: known incorrect sRGB profile
INFO:modules.MapDataset:Loaded mask from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1866_unknown_2/unknown_1866_2_mask.png
INFO:modules.MapDataset:MapDataset created successfully for folder: 1866_unknown_2
Creating MapDataset objects:   1%|          | 1/113 [00:00<00:13,  8.33it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1912_unknown' with author 'unknown' and year '1912'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/L

In [3]:
base_dataset = anchor_map_objects_list


NameError: name 'anchor_map_objects_list' is not defined

In [24]:

target_dataset = target_map_objects_list
target_dataset = [map for map in target_dataset if map.map_info.folder == '1860_pierotti_2' or map.map_info.folder == '1863_pierotti' or map.map_info.folder == '1888_nicole' or map.map_info.folder == '1900_wilson' or map.map_info.folder == '1905_schick' or map.map_info.folder == '1853_zimpel']

In [25]:
with open('./input/base_dataset.pkl', 'wb') as f:
    pickle.dump(base_dataset, f)
    
with open('./input/target_dataset.pkl', 'wb') as f:
    pickle.dump(target_dataset, f)

## 2. First Pairwise Matching

### 2.1 BaseMaps: Orient North and Generate Tensors

In [27]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

In [26]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import modules.MapDataset
import modules.homologous_points_detection

# Process the base_dataset:   

for map_obj in tqdm(base_dataset, desc=f"Base Processing"):
    map_obj.calculate_and_store_north_rotation()
    map_obj.run_superpoint_pipeline()

Loaded SuperGlue model ("outdoor" weights)


Base Processing:   0%|          | 0/86 [00:00<?, ?it/s]INFO:modules.MapDataset:Tensor generated successfully.
INFO:modules.MapDataset:SuperPoint inference completed successfully.
INFO:modules.MapDataset:SuperPoint results stored successfully.
INFO:modules.MapDataset:Mask dilated with buffer: 20
INFO:modules.MapDataset:Filtered keypoints: 1387 remaining after mask removal.
Base Processing:   1%|          | 1/86 [00:02<03:31,  2.49s/it]INFO:modules.MapDataset:Tensor generated successfully.
INFO:modules.MapDataset:SuperPoint inference completed successfully.
INFO:modules.MapDataset:SuperPoint results stored successfully.
INFO:modules.MapDataset:Mask dilated with buffer: 20
INFO:modules.MapDataset:Filtered keypoints: 912 remaining after mask removal.
Base Processing:   2%|▏         | 2/86 [00:03<02:07,  1.52s/it]INFO:modules.MapDataset:Tensor generated successfully.
INFO:modules.MapDataset:SuperPoint inference completed successfully.
INFO:modules.MapDataset:SuperPoint results stored succes

In [28]:
# save the base dataset
import pickle

with open('./input/base_datase_oriented.pkl', 'wb') as f:
    pickle.dump(base_dataset, f)

## to start from here, load the datasets

In [ ]:
import pickle 

with open('./input/base_dataset_oriented.pkl', 'rb') as f:
    base_dataset = pickle.load(f)

with open('./input/target_dataset.pkl', 'rb') as f:
    target_dataset = pickle.load(f)

### 2.2 AnchorMaps: Find Best Match with BaseMaps, Orient North and Generate Tensors

In [ ]:
importlib.reload(modules.homologous_points_detection)
importlib.reload(modules.MapDataset)
from modules.MapDataset import MapDataset
from modules.homologous_points_detection import find_best_matches

In [33]:
from modules.homologous_points_detection import find_single_best_match, estimate_north_rotation
import cv2
import numpy as np
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
from skimage.measure import ransac
from skimage.transform import AffineTransform
import copy
import pickle

    
# Lop over each target map object in the current fold.
for map_obj in tqdm(target_dataset, desc=f"Target Maps Processing"):
    print(map_obj.map_info.folder)
    # Run the SuperPoint pipeline on the target map.
    map_obj.run_superpoint_pipeline()
    best_match = find_single_best_match(map_obj, base_dataset)
    num_matches = len(best_match['superglue_matches_df'])
    initial_num_matches = num_matches
        
    min_match_score = 0.3
    rotation_angles = [np.pi / 2, np.pi, 3 * np.pi / 2]
    rotation_degrees = [90, 180, 270]
        
        # If the current number of matches is less than 100, attempt imposed rotations.
    if num_matches < 100:
        best_num_matches = num_matches
        best_angle_rad = None

        for angle_rad, angle_deg in zip(rotation_angles, rotation_degrees):
            map_obj.north_rotation_angle = angle_rad
            map_obj.run_superpoint_pipeline()
            best_match_rotated = find_single_best_match(map_obj, base_dataset)
            new_num_matches = len(best_match_rotated['superglue_matches_df'])
            if new_num_matches > best_num_matches:
                best_num_matches = new_num_matches
                best_angle_rad = angle_rad
                best_match = best_match_rotated

        if best_angle_rad is not None:
            map_obj.north_rotation_angle = best_angle_rad
            map_obj.run_superpoint_pipeline()
            best_match = best_match_rotated
            #print(f"Imposed rotation of {np.degrees(best_angle_rad):.2f}° improved matches from {num_matches} to {best_num_matches}.")
            derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=True)
            #print('Derived north rotation (deg):', np.degrees(derived_north_rotation))
        else:
            map_obj.north_rotation_angle = 0.0
            map_obj.run_superpoint_pipeline()
            best_match = find_single_best_match(map_obj, base_dataset)
            derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=False)
            #print(f"No imposed rotation improved matches from {num_matches}. Calculated north rotation: {np.degrees(derived_north_rotation):.2f}°.")
    else:
        # If there are enough matches, directly estimate the north rotation.
        derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=False)

    map_obj.north_rotation_angle = derived_north_rotation
    map_obj.run_superpoint_pipeline()



Target Maps Processing:   0%|          | 0/6 [00:00<?, ?it/s]

1905_schick


ERROR:modules.MapDataset:Failed to preprocess image: Image and mask sizes do not match: (11733, 8522) vs (15000, 10895)
Target Maps Processing:   0%|          | 0/6 [00:00<?, ?it/s]


ValueError: Image and mask sizes do not match: (11733, 8522) vs (15000, 10895)

In [ ]:
# we can store the oriented target maps
with open('./input/target_dataset_oriented.pkl', 'wb') as f:
    pickle.dump(target_dataset, f)

This part of the process is quadratic, the more maps we have the more this becomes time consuming. We will need to find a way to optimize this part of the process. Maybe just keeping the n best matches to consider for the steps after?

### 2.3 Repeat Pairwise Matching with the new tensors
We compare each map of the target dataset with each map of the anchor dataset and store all the matches. 

In [ ]:
from modules.homologous_points_detection import  find_best_matches
from modules.MapDataset import MapDataset
from typing import List

#in Evaluation mode we can collect more than one match per map:
find_best_matches(target_dataset, base_dataset, threshold_number_matches=200, number_best_results=3, evaluation = True, min_score=0.1 )


## 3. Match Improvements

### 3.1 Ransac + Delaunay Cleaning
#### RANSAC

In [ ]:
from modules.homologous_points_detection import remove_outliers_ransac
from tqdm import tqdm

for map_obj in tqdm(target_dataset):
    for best_match in map_obj.best_matches_result:
        matches_df = best_match.superglue_matches
        filtered_matches_df = remove_outliers_ransac(matches_df, ransac_threshold=20, ransac_max_trials=1000)
        best_match.ransac_filtered_matches = filtered_matches_df
        

In [ ]:

import modules.visualization
importlib.reload(modules.visualization)
from modules.visualization import plot_delaunay_matches

# find the map obj whose map_info.folder is '1860_pierotti_2'
map_obj = [map_obj for map_obj in target_dataset if map_obj.map_info.folder == '1860_pierotti_2'][0]
for best_match in map_obj.best_matches_result[:2]:
    plot_delaunay_matches(best_match, best_match.superglue_matches, best_match.ransac_filtered_matches, map_obj, base_dataset)

#### Delaunay

In [ ]:

# let's plot the improvement obtained by the Delaunay filterd
from modules.visualization import plot_delaunay_matches
from modules.homologous_points_detection import filter_match_with_delaunay   
import matplotlib.pyplot as plt
from tqdm import tqdm
# Suppress the specific warning about invalid values in intersection
import warnings
warnings.filterwarnings("ignore", message="invalid value encountered in intersection")
import numpy as np

for map_obj in tqdm(target_dataset):
    for best_match in map_obj.best_matches_result:
        best_match.delaunay_filtered_matches = filter_match_with_delaunay(map_obj, base_dataset, best_match, best_match.ransac_filtered_matches, similarity_threshold=0.6, min_score_match=0.2)


### 3.2 Points Addition

### 3.3 Ransac + Delaunay Cleaning

## 4. GCP transfer


## 5. Results Reliability

## Saving the results: go over each folder of the target maps, create a folder and save the results.

Future directions: using a recursive technique: only propagating georef to maps in which we are certain and then use those as anchors themselves